In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
# Install required libraries
# pip install datasets transformers sentence-transformers torch

import pandas as pd
import numpy as np
from datasets import load_dataset, Dataset
import torch
from transformers import (
    AutoTokenizer, AutoModel, pipeline,
    BertTokenizer, BertModel
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity as sklearn_cosine
from sentence_transformers import SentenceTransformer, util

# Load data
df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')

In [3]:
# ============================================================
# Q1: Load with HuggingFace datasets, create combined_text
# ============================================================
dataset = Dataset.from_pandas(df)

def add_combined_text(example):
    example['combined_text'] = str(example['prompt']) + ' ' + str(example['A'])
    return example

dataset = dataset.map(add_combined_text)

# Character length at index 51
print("combined_text at index 51:")
print(repr(dataset[51]['combined_text']))
print("Length:", len(dataset[51]['combined_text']))

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

combined_text at index 51:
'Determine the correct option: What is the reason behind the designation of Class L dwarfs, and what is their color and composition? among the listed options. Class L dwarfs are hotter than M stars and are designated L because L is the remaining letter alphabetically closest to M. They are bright blue in color and are brightest in ultraviolet. Their atmosphere is hot enough to allow metal hydrides and alkali metals to be prominent in their spectra. Some of these objects have masses large enough to support hydrogen fusion and are therefore stars, but most are of substellar mass and are therefore brown dwarfs.'
Length: 614


In [4]:
# ============================================================
# Q2: BERT tokenizer vocabulary size
# ============================================================
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
print("Vocab size:", tokenizer.vocab_size)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Vocab size: 30522


In [5]:
# ============================================================
# Q3: [SEP] token ID
# ============================================================
sep_id = tokenizer.convert_tokens_to_ids('[SEP]')
print("[SEP] token ID:", sep_id)

[SEP] token ID: 102


In [6]:
# ============================================================
# Q4: Tokenize entire prompt column → shape of input_ids
# ============================================================
prompts = [str(p) for p in df['prompt'].tolist()]

encoded = tokenizer(
    prompts,
    padding='max_length',
    truncation=True,
    max_length=128,
    return_tensors='pt'
)

print("input_ids shape:", encoded['input_ids'].shape)

input_ids shape: torch.Size([2000, 128])


In [7]:
# ============================================================
# Q5: Attention head dimensionality
# ============================================================
hidden_size = 768
num_heads = 12
head_dim = hidden_size // num_heads
print("Each attention head dimension:", head_dim)

Each attention head dimension: 64


In [8]:
# ============================================================
# Q6: Shape of last_hidden_state for row ID 0
# ============================================================
model = AutoModel.from_pretrained('bert-base-uncased')
model.eval()

row0_prompt = str(df.iloc[0]['prompt'])
inputs = tokenizer(row0_prompt, return_tensors='pt')

with torch.no_grad():
    outputs = model(**inputs)

print("last_hidden_state shape:", outputs.last_hidden_state.shape)
print("Number of tokens:", inputs['input_ids'].shape[1])

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


last_hidden_state shape: torch.Size([1, 31, 768])
Number of tokens: 31


In [9]:
# ============================================================
# Q7: Sum of first 5 values of [CLS] token embedding
# ============================================================
cls_embedding = outputs.last_hidden_state[0, 0, :]  # [CLS] is index 0
first_5 = cls_embedding[:5]
print("First 5 CLS values:", first_5)
print("Sum of first 5:", round(first_5.sum().item(), 4))

First 5 CLS values: tensor([-0.4677, -0.0754, -0.2019, -0.0071, -0.4480])
Sum of first 5: -1.2001


In [10]:
# ============================================================
# Q8: Attention weight [CLS] → 'fusion' in "Light-ion fusion is a technique."
# ============================================================
model_attn = AutoModel.from_pretrained('bert-base-uncased', output_attentions=True)
model_attn.eval()

text = "Light-ion fusion is a technique."
inputs2 = tokenizer(text, return_tensors='pt')

# Check tokens
tokens = tokenizer.convert_ids_to_tokens(inputs2['input_ids'][0])
print("Tokens:", tokens)
fusion_idx = tokens.index('fusion')
print("Index of 'fusion':", fusion_idx)

with torch.no_grad():
    outputs2 = model_attn(**inputs2)

# Last layer (index -1), first head (index 0)
# attentions shape: (num_layers, batch, num_heads, seq_len, seq_len)
last_layer_attn = outputs2.attentions[-1]  # (1, 12, seq_len, seq_len)
head0_attn = last_layer_attn[0, 0]         # (seq_len, seq_len)

# [CLS] (index 0) attention to 'fusion'
cls_to_fusion = head0_attn[0, fusion_idx].item()
print(f"[CLS] → fusion attention weight: {cls_to_fusion:.4f}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Tokens: ['[CLS]', 'light', '-', 'ion', 'fusion', 'is', 'a', 'technique', '.', '[SEP]']
Index of 'fusion': 4
[CLS] → fusion attention weight: 0.1025


In [11]:
# ============================================================
# Q9: Sentence-Transformers cosine similarity for row ID 0
# ============================================================
st_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

row0 = df.iloc[0]
prompt_emb = st_model.encode(str(row0['prompt']), convert_to_tensor=True)
optB_emb   = st_model.encode(str(row0['B']),      convert_to_tensor=True)

sim = util.cos_sim(prompt_emb, optB_emb)
print(f"Cosine similarity (prompt vs B, row 0): {sim.item():.4f}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Cosine similarity (prompt vs B, row 0): 0.7658


In [12]:
# ============================================================
# Q10: MAP@3 — MiniLM pipeline vs TF-IDF pipeline
# ============================================================
options = ['A', 'B', 'C', 'D', 'E']

def map_at_k(truth, preds, k=3):
    score, hits = 0.0, 0
    for i, p in enumerate(preds[:k], 1):
        if p == truth:
            hits += 1
            score += hits / i
    return score

# --- TF-IDF Pipeline (from Milestone 1) ---
all_texts = []
for _, row in df.iterrows():
    all_texts.append(str(row['prompt']))
    for opt in options:
        all_texts.append(str(row[opt]))

tfidf = TfidfVectorizer(stop_words='english')
tfidf.fit(all_texts)

def tfidf_top3(row):
    pv = tfidf.transform([str(row['prompt'])])
    sims = {opt: sklearn_cosine(pv, tfidf.transform([str(row[opt])]))[0][0]
            for opt in options}
    return sorted(sims, key=sims.get, reverse=True)[:3]

df['tfidf_top3'] = df.apply(tfidf_top3, axis=1)
tfidf_map = df.apply(lambda r: map_at_k(r['answer'], r['tfidf_top3']), axis=1).mean()
print(f"TF-IDF MAP@3: {tfidf_map:.4f}")

# --- MiniLM Pipeline ---
print("Encoding all prompts and options with MiniLM...")

prompt_embs = st_model.encode(df['prompt'].tolist(), batch_size=64,
                               show_progress_bar=True, convert_to_tensor=True)
opt_embs = {
    opt: st_model.encode(df[opt].tolist(), batch_size=64,
                         show_progress_bar=True, convert_to_tensor=True)
    for opt in options
}

def minilm_top3(idx):
    sims = {opt: util.cos_sim(prompt_embs[idx], opt_embs[opt][idx]).item()
            for opt in options}
    return sorted(sims, key=sims.get, reverse=True)[:3]

df['minilm_top3'] = [minilm_top3(i) for i in range(len(df))]
minilm_map = df.apply(lambda r: map_at_k(r['answer'], r['minilm_top3']), axis=1).mean()
print(f"MiniLM MAP@3: {minilm_map:.4f}")

# Count: correct NOT in TF-IDF top3 BUT IS in MiniLM top3
count = 0
for _, row in df.iterrows():
    ans = row['answer']
    if ans not in row['tfidf_top3'] and ans in row['minilm_top3']:
        count += 1
print(f"Correct in MiniLM but NOT in TF-IDF top3: {count}")

TF-IDF MAP@3: 0.3119
Encoding all prompts and options with MiniLM...


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

MiniLM MAP@3: 0.4231
Correct in MiniLM but NOT in TF-IDF top3: 462


In [13]:
# ============================================================
# Q11: Zero-shot classification (Softmax) — index 1, options A,B,C
# ============================================================
zs_clf = pipeline("zero-shot-classification")  # facebook/bart-large-mnli

row1 = df.iloc[1]
prompt1 = str(row1['prompt'])
candidates = [str(row1['A']), str(row1['B']), str(row1['C'])]

result_softmax = zs_clf(prompt1, candidate_labels=candidates)
print("Softmax result:", result_softmax)
print("Top label:", result_softmax['labels'][0])
print("Top score:", round(result_softmax['scores'][0], 4))
print("Sum of scores:", sum(result_softmax['scores']))

No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Softmax result: {'sequence': 'What is accelerator-based light-ion fusion?', 'labels': ['Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively easy to implement and can be done in an efficient manner, requiring only a vacuum tube, a pair of electrodes, and a high-voltage transformer. Fusion can be observed with as little as 10 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively difficult to implement and requires a complex system of vacuum tubes, electrodes, and transformers. Fusion can be observed with as little as 100 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to 

In [14]:
# ============================================================
# Q12: Zero-shot with multi_label=True (Sigmoid)
# ============================================================
result_sigmoid = zs_clf(prompt1, candidate_labels=candidates, multi_label=True)
print("Sigmoid result:", result_sigmoid)

sum_softmax = sum(result_softmax['scores'])
sum_sigmoid = sum(result_sigmoid['scores'])
print(f"Sum softmax: {sum_softmax:.4f}")
print(f"Sum sigmoid: {sum_sigmoid:.4f}")
print(f"Absolute difference: {abs(sum_softmax - sum_sigmoid):.4f}")

Sigmoid result: {'sequence': 'What is accelerator-based light-ion fusion?', 'labels': ['Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively easy to implement and can be done in an efficient manner, requiring only a vacuum tube, a pair of electrodes, and a high-voltage transformer. Fusion can be observed with as little as 10 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce heavy-ion fusion reactions. This method is relatively difficult to implement and requires a complex system of vacuum tubes, electrodes, and transformers. Fusion can be observed with as little as 10 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to i

In [16]:
# ============================================================
# Q13: Generative AI with flan-t5-small for row index 0
# ============================================================
gen_pipeline = pipeline(
    task="text-generation",
    model="google/flan-t5-small"
)

row0 = df.iloc[0]
prompt_str = (
    f"Question: {row0['prompt']}. "
    f"Is the correct answer A: {row0['A']} or B: {row0['B']}? "
    f"Answer with just the letter A or B."
)

print("Input prompt:", prompt_str)
output = gen_pipeline(prompt_str, max_new_tokens=5)
print("Model output:", output[0]['generated_text'])

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'Ernie4_5ForCausalLM', 'Ernie4_5_MoeForCausalLM', 'Exaone4ForCausalLM', 'FalconForCausalLM', 'FalconH1ForCausalLM', 'FalconMambaForCausa

Input prompt: Question: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.. Is the correct answer A: Martin Heidegger believes that humans exist within a time continuum that is infinite and does not have a defined beginning or end. The relationship to the past involves acknowledging it as a historical era, and the relationship to the future involves creating a world that will endure beyond one's own time. or B: Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.? Answer with just the letter A or B.
Model output: Question: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.. Is the correct answer A: Mart

In [17]:
# import pandas as pd

# # Read the CSV file
df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")

# Save an identical copy
df.to_csv("submission.csv", index=False)

print("File copied successfully.")

File copied successfully.
